# DeepLearnX: Deep Learning-Based Student Concept Understanding & Knowledge Gap Detection
### 3rd-Year B.Tech Artificial Intelligence & Data Science Capstone Project

**Project Objectives:**
1. Move beyond traditional mark-based evaluation to detect **Genuine Mastery** vs **False Mastery** (rote memorization).
2. Isolate active **Conceptual Misconceptions** from distractor response patterns.
3. Traverse a **Directed Acyclic Graph (DAG)** of concept dependencies to uncover root **Prerequisite Knowledge Gaps**.
4. Formulate personalized remediation pathways.

In [ ]:
import os
import sys
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from pathlib import Path

# Set project root paths
PROJECT_ROOT = Path("..").resolve()
sys.path.insert(0, str(PROJECT_ROOT / "model"))
sys.path.insert(0, str(PROJECT_ROOT / "backend" / "app"))

from architecture import DeepCognitiveNet
from predictor import DeepLearnXPredictor
from graph_engine import KnowledgeGraphEngine

print("PyTorch Version:", torch.__version__)
print("Environment initialized successfully.")

## 1. Dataset Inspection & Cognitive Features
Let's load the generated student session dataset and explore the cognitive features.

In [ ]:
df_train = pd.read_csv(PROJECT_ROOT / "dataset" / "student_responses_train.csv")
print(f"Training set samples: {len(df_train)}")
df_train.head()

In [ ]:
# Analyze Cognitive Divergence across classes
df_train.groupby("state_name")[["acc_overall", "acc_basic", "acc_conc", "acc_app", "tier_divergence"]].mean()

### Notice the Key Pattern:
- **False Mastery:** `acc_basic` is very high (~92%), `acc_conc` is ~82%, but `acc_app` collapses to ~18%, yielding a large positive **Tier Divergence** (>0.74).
- **Genuine Mastery:** Consistently high across all three tiers (>0.85), with minimal divergence.
- **Knowledge Gap:** Fails uniformly across basic, conceptual, and application tiers.

## 2. Model Architecture & Evaluation Metrics
Loading the saved training history and test evaluation metrics.

In [ ]:
with open(PROJECT_ROOT / "model" / "metrics.json", "r") as f:
    metrics = json.load(f)

print("Test Set Evaluation Results:")
print(f"  Accuracy:        {metrics['accuracy'] * 100:.2f}%")
print(f"  Macro Precision: {metrics['precision_macro'] * 100:.2f}%")
print(f"  Macro Recall:    {metrics['recall_macro'] * 100:.2f}%")
print(f"  Macro F1-Score:  {metrics['f1_macro'] * 100:.2f}%")
print("\nPer-Class Breakdown:")
for c_name, vals in metrics["per_class"].items():
    print(f"  {c_name:<26} -> F1: {vals['f1_score']:.4f} (Support: {vals['support']})")

In [ ]:
# Plot Training Loss Convergence Curves
with open(PROJECT_ROOT / "model" / "training_history.json", "r") as f:
    history = json.load(f)

plt.figure(figsize=(9, 4))
plt.plot(history["train_loss"], label="Training Loss (Multi-Task)", marker="o")
plt.plot(history["val_loss"], label="Validation Loss", marker="s")
plt.title("DeepLearnX DeepCognitiveNet Loss Curve")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 3. End-to-End Inference Verification
Testing False Mastery detection on a simulated student.

In [ ]:
predictor = DeepLearnXPredictor()

# Simulating False Mastery scenario: 100% on Basic, 0% on Application
test_responses = [
    {"question_id": "q_ann_basic_1", "selected_option": "A", "response_time": 18.0, "confidence": 5},
    {"question_id": "q_ann_concept_1", "selected_option": "A", "response_time": 22.0, "confidence": 5},
    {"question_id": "q_ann_app_1", "selected_option": "C", "response_time": 65.0, "confidence": 3},
    {"question_id": "q_act_basic_1", "selected_option": "A", "response_time": 14.0, "confidence": 5},
    {"question_id": "q_act_concept_1", "selected_option": "A", "response_time": 25.0, "confidence": 4},
    {"question_id": "q_act_app_1", "selected_option": "B", "response_time": 70.0, "confidence": 3}
]

analysis = predictor.analyze_session(test_responses)
print("Predicted Understanding State:", analysis["predicted_state"])
print("Model Confidence:             ", f"{analysis['confidence'] * 100:.1f}%")
print("False Mastery Diagnosis:      ", analysis["false_mastery_breakdown"]["diagnosis"])
print("Tier Divergence (Basic-App):   ", f"{analysis['false_mastery_breakdown']['tier_divergence']}%")

## 4. Knowledge Dependency DAG Traversal
Tracing root prerequisite gap using the KnowledgeGraphEngine.

In [ ]:
engine = KnowledgeGraphEngine()
concept_scores = {
    "backpropagation": {"correct": 0, "total": 2, "acc": 0.0},
    "chain_rule": {"correct": 0, "total": 1, "acc": 0.0},
    "derivatives": {"correct": 1, "total": 1, "acc": 1.0}
}
graph_eval = engine.evaluate_student_graph(concept_scores, [], "Knowledge Gap")
print("Primary Prerequisite Bottleneck:", graph_eval["primary_gap_name"])
print("Root Prerequisite Gaps:         ", graph_eval["root_prerequisite_gaps"])